In [2]:
%load_ext autoreload
%autoreload 2

# Using `rxn_ca` with Jobflow

While the simple demostration from the `basic_usage.ipynb` is great for small scale studies and debugging, most applications of this package deal with rapid iteration of either a reaction recipe, a reaction library (through the scoring function), or both. In this notebook, we'll see how to run *many* reactions in parallel, making use of modern HPC infrastructure. Obviously, the method shown here is only useful if you're going to be running _several_ simulations (atleast of the order of hundreds). If not, stick to what the basic guide has.

We'll use the same $MgAl_{2}O_{4}$ example as earlier. 

In [3]:
reactants = {
    "MgO": 1,
    "Al2O3": 1,
}

As before, we'll follow the basic usage notebook and set-up everything we need to run a reaction:

In [4]:
from rxn_ca.core.recipe import ReactionRecipe
from rxn_ca.core.heating import HeatingSchedule, HeatingStep
from rxn_ca.utilities.get_entries import get_entries

heating_schedule = HeatingSchedule.build(
    HeatingStep.sweep(500, 1600, stage_length = 1, temp_step_size=50),
    HeatingStep.hold(1600, stage_length = 20)
)

recipe = ReactionRecipe(
    reactant_amounts=reactants,
    heating_schedule=heating_schedule
)

chem_sys = "Mg-Al-O" # Scope to phases containing one or more of (Mg, Al, O)
metastability_cutoff = 0.03
exclude_theoretical = True

# Download phases in this chemical system
all_entries = get_entries(
    chem_sys=chem_sys,
    metastability_cutoff = metastability_cutoff,
    exclude_theoretical_phases = exclude_theoretical
)

print(f"Considering {len(all_entries)} phases after filtering by observation and metastability.")

/Users/virkaran/miniconda3/envs/rxn-ca-dev/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning:

IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html

2025-12-02 11:26:52,761	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
Retrieving ThermoDoc documents: 100%|██████████| 249/249 [00:00<00:00, 2198698.31it/s]


Retrieving entry volumes...
Retrieving entry obs....


Retrieving SummaryDoc documents: 100%|██████████| 207/207 [00:00<00:00, 4693086.10it/s]


Retrieving entry mps...
Considering 9 phases after filtering by observation and metastability.


In [ ]:
from rxn_network.enumerators.utils import run_enumerators
from rxn_network.enumerators.basic import BasicEnumerator
from rxn_network.enumerators.minimize import MinimizeGibbsEnumerator


gibbs_enums = [MinimizeGibbsEnumerator(), BasicEnumerator()]
rxn_set = run_enumerators(gibbs_enums, all_entries)

all_temps = recipe.heating_schedule.all_temps
print(f"All temperatures required: {all_temps}")
temp_rxn_mapping = rxn_set.compute_at_temperatures(all_temps);

In [6]:
from rxn_ca.utilities.get_scored_rxns import get_scored_rxns
from rxn_ca.phases import SolidPhaseSet

solid_phase_set = SolidPhaseSet.from_entry_set(all_entries)

reaction_library = get_scored_rxns(
    rxn_set, 
    temps=all_temps,
    phase_set=solid_phase_set, 
    rxns_at_temps=temp_rxn_mapping,
    parallel=True
)

Retrieving entry volumes...
Retrieving entry obs....


Retrieving SummaryDoc documents: 100%|██████████| 184/184 [00:00<00:00, 3998714.69it/s]


Retrieving entry mps...


Scoring reactions... at temp 900: 128it [00:00, 1711.14it/s]
Scoring reactions... at temp 1550: 128it [00:00, 1560.98it/s]
Scoring reactions... at temp 1300: 128it [00:00, 1732.05it/s]
Scoring reactions... at temp 1050: 128it [00:00, 1589.35it/s]
Scoring reactions... at temp 650: 128it [00:00, 1451.31it/s]
Scoring reactions... at temp 550: 128it [00:00, 1629.26it/s]
Scoring reactions... at temp 800: 128it [00:00, 1526.94it/s]
Scoring reactions... at temp 1450: 128it [00:00, 1602.16it/s]
Scoring reactions... at temp 950: 128it [00:00, 1612.31it/s]
Scoring reactions... at temp 700: 128it [00:00, 1606.10it/s]
Scoring reactions... at temp 1200: 128it [00:00, 1322.77it/s]
Scoring reactions... at temp 1600: 128it [00:00, 1467.57it/s]
Scoring reactions... at temp 1100: 128it [00:00, 2266.22it/s]
Scoring reactions... at temp 1350: 128it [00:00, 1995.14it/s]
Scoring reactions... at temp 600: 128it [00:00, 2031.26it/s]
Scoring reactions... at temp 1250: 128it [00:00, 2147.54it/s]
Scoring reactio

Notice how we're not writing anything to disk? ;)

# Defining the Maker

Jobflow allows us to conveniently make several "jobs" that are really similar, and to orchestrate complex workflows consisting of several "jobs". Refer to the[jobflow docs](https://materialsproject.github.io/jobflow/index.html) to set-up eveything you need to use it. Jobflow also gives us access to workflow managers such as [FireWorks](https://materialsproject.github.io/fireworks/) to run the workflows we make on an HPC. 

In [7]:
from rxn_ca.computing.jobs import MultiRxnCAMaker, MultiRxnType

Everything we need is in the MultiRxnCAMaker. There are 3 types of parallel simulations we can run:
1. One reaction library, many recipes
2. One recipe, many reaction libraries
3. Many recipes and many reaction libraries

To switch between these, use the `MultiRxnType` enum. The maker defaults to the 3rd case. Here is an example of how to define the maker and make the jobs. In this case we're simply reciplicating the recipe and library we made earlier, but the same holds if you make several of these objects. 

In [8]:
num_replications = 3
# case 1
maker = MultiRxnCAMaker(multi_rxn_type=MultiRxnType.MULTI_RECIPE)
job = maker.make(recipes=[recipe] * num_replications, reaction_libraries=reaction_library)

# case 2
maker = MultiRxnCAMaker(multi_rxn_type=MultiRxnType.MULTI_LIB)
job = maker.make(recipes=recipe, reaction_libraries=[reaction_library] * num_replications)

# case 3
maker = MultiRxnCAMaker(multi_rxn_type=MultiRxnType.MULTI_LIB_AND_RECIPE)
job = maker.make(recipes=[recipe] * num_replications, reaction_libraries=[reaction_library] * num_replications)

Now we have two options for how to run the job:
1. Locally
2. Through a workflow manager (like FireWorks)

Here's how you do #1:

In [9]:
from jobflow import run_locally

output = run_locally(job, create_folders=True)

2025-12-02 11:27:10,845 INFO Started executing jobs locally
2025-12-02 11:27:11,102 INFO Starting job - MultiRxnCA Maker (9d96d998-acdc-4f9c-8353-05b86ef67e8a)
Using 1 CPU(s) per task (total: 9 realizations, 12 CPUs available)
================= RUNNING 9 REALIZATIONS ACROSS 3 RECIPES ON 12.0 CORES =================
Distributing tasks across available cores...
Recipe 0: 3/3 successful realizations
Recipe 1: 3/3 successful realizations
Recipe 2: 3/3 successful realizations
============ SAVING RESULT DOCS =============
Saving result doc 0 to file...
Saving result doc 1 to file...
Saving result doc 2 to file...
============ CREATING MULTI RXN CA RESULT DOCUMENT =============


Constructing result from diffs: 100%|██████████| 499/499 [00:03<00:00, 136.27it/s]


3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
2025-12-02 11:31:58,421 INFO Finished job - MultiRxnCA Maker (9d96d998-acdc-4f9c-8353-05b86ef67e8a)
2025-12-02 11:31:58,423 INFO Finished executing jobs locally


And just like that, all 3 replications of this recipe/library combination have run in parallel. To be precise, we ran 9 simulations, since each simulation runs 3 replicates by default to sample the system better. 

By default, each of the 9 simulations uses 1 core or 1 cpu. Anything more does not improve performance by a whole lot. Hence, you should try to run as many (or more) simulations as the number of cores your compute node has access to. If you run into memory errors, consider increasing the `cpus_per_task` arg of the maker to 2 (or even 4 for _large_ and _beefy_ simulations). 

By default, the reaction output documents are stored in the directory where the job is run, and useful analysis plots are built and stored in jobflow's JobStore, which allows you to query these from any system with the Store configured (refer to the Jobflow docs for what this means). 

In [11]:
job_output = output[job.uuid][1].output

The analysis plots are in this `job_outputs` object in a serialized format. We can deserialize and view them with plotly:

In [12]:
from plotly.io import from_json

for i, serial_plot in enumerate(job_output.molar_fraction_plots):
    print(f"Plotting job {i}...")
    fig = from_json(serial_plot['figure_json'])
    fig.show()

Plotting job 0...


Plotting job 1...


Plotting job 2...


As we can see, the results from the 3 replications are practically identical (yay!). You can similarly plot the mass/volume/elemental fractions or amounts. 

We can also get the raw data used to make these plots:

In [14]:
for i, serial_plot in enumerate(job_output.molar_fraction_plots):
    print(f"Plotting job {i}...")
    for phase in serial_plot['data'].keys():
        print(phase)
        print(serial_plot['data'][phase]['y'])
        print()
    break

Plotting job 0...
Temperature
[500, 500, 550, 550, 600, 600, 650, 650, 700, 700, 750, 750, 800, 800, 850, 850, 900, 900, 950, 950, 1000, 1000, 1050, 1050, 1100, 1100, 1150, 1150, 1200, 1200, 1250, 1250, 1300, 1300, 1350, 1350, 1400, 1400, 1450, 1450, 1500, 1500, 1550, 1550, 1600, 1600, 1600, 1600]

MgAl2O4
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 9.878388080160889e-05, 9.878388080160889e-05, 9.878388080160889e-05, 9.878388080160889e-05, 9.878388080160889e-05, 9.878388080160889e-05, 0.00014817904879559112, 0.00014817904879559112, 0.00014817904879559112, 0.00024699811902670425, 0.00024699811902670425, 0.00034584363179496013, 0.00034584363179496013, 0.00034584363179496013, 0.00034584363179496013, 0.00034584363179496013, 0.00034584363179496013, 0.00034584363179496013, 0.00044471559771523276, 0.0004941838002055926, 0.0004941838002055926, 0.0004941838002055926, 0.0005436140274080788, 0.0005436140274080788, 0.0006425389314997408, 0.0006425389314997408, 0.0006425389314

Finally, you can also see if the simulation has converged:

In [15]:
job_output.have_simulations_converged

[False, False, False]

Clearly, the really short simulation we ran did not converge (i.e., there's still things that can happen!), and a longer simulation time is needed to fully determine the path of this reaction. For a more thorough analysis, you'll have to read in the actual results doc files, whose location is stored under `job_output.run_dir`. The index of the output plots and files is the same order as the recipes or libraries. 

In [19]:
#for fireworks
#from jobflow.manager.fireworks import flow_to_workflow
#from fireworks.core.launchpad import LaunchPad
#lp = LaunchPad.from_file("fw_config.yaml")
#wf = flow_to_workflow(job)
#lp.add_wf(wf)

#then, go to your HPC and run the workflow with:
#qlaunch singleshot -f <fw_id>
#or
#rlaunch singleshot -f <fw_id>
#or
#qlaunch rapidfire --nlaunches <num_launches> (for many workflows in parallel)